# Qwen3-1.7B CPT trên Modal A100 80GB

Dùng notebook kernel **CPU**; launcher cấp riêng một A100 80GB. Tạo Secret `hf-token` chứa `HF_TOKEN` trong cùng Modal environment trước khi chạy. Full fine-tuning: 2 epoch, batch 16, accumulation 2, context 2048. Chỉ bắt đầu một job cho mỗi run ID.


In [ ]:
# Code nguồn đóng gói trong notebook; không chứa token hoặc checkpoint.
import base64, hashlib, io, os, zipfile
from pathlib import Path
payload = base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest() == '3906d5a1a5b99aa94a567cccdf498ff642c5dfdb36d4a87d2d15a00561451640'
work = Path.cwd() / "cpt_workspace"
work.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        target = (work / item.filename).resolve()
        if not target.is_relative_to(work.resolve()):
            raise ValueError("Unsafe archive path")
    archive.extractall(work)
os.chdir(work / "continue-pretrainedllms")
print("Project:", Path.cwd())


In [ ]:
%pip install 'modal>=1,<2'


## Đăng nhập Modal CLI

Chạy một lần nếu CLI chưa được xác thực; mở liên kết được in ra và đăng nhập đúng workspace/environment chứa Secret.


In [ ]:
import subprocess
def modal_command(*arguments):
    subprocess.run(["modal", *arguments], check=True)

modal_command("setup")


## Kiểm tra CPU trước khi thuê GPU

Không tải corpus hoặc weights Qwen. Nếu cell lỗi, xử lý lỗi trước khi chạy tiếp.


In [ ]:
modal_command("run", "scripts/modal_full_ft.py", "--check-only")


## Chuẩn bị corpus trên CPU

Tải 253 shard từ `iraprp/chayqwen`, ghim model/data revision và lưu packed Arrow trên Volume. Có thể mất nhiều giờ ở lần đầu.


In [ ]:
modal_command("run", "scripts/modal_full_ft.py", "--prepare-only")


## Train hoặc resume một phiên

Một phiên dừng có lưu checkpoint sau khoảng 18 giờ để chừa thời gian trong giới hạn 24 giờ. `--detach` giữ job remote nếu notebook mất kết nối; có thể dừng cell sau khi Modal xác nhận job đã khởi chạy. Chờ phiên trước kết thúc trước khi chạy lại. Giữ nguyên notebook/code/config để resume.


In [ ]:
RUN_ID = 'qwen3-1.7b-a10080-e02-b16-ga2-v2'
modal_command("run", "--detach", "scripts/modal_full_ft.py", "--run-id", RUN_ID)


## Xem trạng thái và ETA

Có thể chạy từ notebook khác hoặc sau khi phiên kết thúc. ETA lấy từ tốc độ đo được, gồm overhead xảy ra giữa các lần log.


In [ ]:
RUN_ID = 'qwen3-1.7b-a10080-e02-b16-ga2-v2'
modal_command("run", "scripts/modal_full_ft.py", "--status-only", "--run-id", RUN_ID)


## Tải hồ sơ thí nghiệm

Gói này chứa log, thống kê, cấu hình và snapshot source; không chứa model weights. Tải `audit_bundle.zip` về máy qua Files panel.


In [ ]:
modal_command("run", "scripts/modal_full_ft.py", "--export-only", "--run-id", RUN_ID)
modal_command("volume", "get", "qwen3-1-7b-full-ft-runs", f"/{RUN_ID}/audit_bundle.zip", "./audit_bundle.zip")
